In [ ]:
#!pip install xarray
#!pip install scipy
#!pip install cdsapi

: 

#### Import Libraries

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import marineHeatWaves as mhw
import datetime as dt
import math

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import griddata
from matplotlib.gridspec import GridSpec

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

#### Define Slice Function

The coordinates have to be defined in 0-360 format.

In [ ]:
def spatial_subset(ds):
    out = ds.sel(lon = slice(350.0, 355.0),
                 lat = slice(36.0, 42.3))
    return out

## Get Baseline

The baselilne used is 1991-2021

In [ ]:
baseline_df = pd.DataFrame()
for year in range(1981, 2000):

    var = xr.open_dataset(f"oisst_daily/sst.day.mean.{year}.nc", 
                        engine="netcdf4", decode_times=True)
    
    print(f'Opening {year}')
    
    var_sel = spatial_subset(var)
    
    var_sst = var_sel["sst"]

    baseline_df = pd.concat([baseline_df, var_sst.to_dataframe()])
    
del var

## Years to be Analysed

In [ ]:
y2000 = xr.open_dataset("oisst_daily/sst.day.mean.2000.nc", 
                        engine="netcdf4", decode_times=True)

#### Slice datasets' coordinates

In [ ]:
y2000_sel = spatial_subset(y2000)
y2000_sst = y2000_sel["sst"]
y2000_time = y2000_sel["time"]

In [ ]:
baseline_df = baseline_df.reset_index()

y2000_df = y2000_sst.to_dataframe().reset_index()


## Conversion to ordinal time

In [ ]:
def to_ordinal(time_array):
    ordinals = []

    for t in time_array:
        # Case 1: numpy.datetime64
        if isinstance(t, np.datetime64):
            ts = pd.Timestamp(t)
            ordinals.append(ts.to_pydatetime().date().toordinal())

        # Case 2: cftime objects
        else:
            ordinals.append(dt.date(t.year, t.month, t.day).toordinal())

    return np.array(ordinals)

## Detect MHWs

In [ ]:
y2000_time_ord = to_ordinal(y2000_df['time'])
baseline_time_ord = to_ordinal(baseline_df["time"])

In [ ]:
results = []

for (lat, lon), group in y2000_df.groupby(["lat", "lon"]):

    # Subset baseline dataframe for this grid point
    baseline_point = (
    baseline_df
    .query("lat == @lat and lon == @lon")
    .sort_values("time")
)

    # Subset baseline time ordinals USING THE SAME ROWS
    baseline_idx = baseline_point.index.values
    baseline_time_ord_sub = baseline_time_ord[baseline_idx]
    baseline_temp = baseline_point["sst"].values


    t_ord = to_ordinal(group["time"])
    temp = group["sst"].values

    mhw_events, clim = mhw.detect(
    t_ord,
    temp,
    climatologyPeriod=[1981, 2010],
    alternateClimatology=[baseline_time_ord_sub, baseline_temp]
)


    mhw_df = pd.DataFrame(mhw_events)
    mhw_df["lat"] = lat
    mhw_df["lon"] = lon

    results.append(mhw_df)

mhw_all = pd.concat(results, ignore_index=True)
mhw_all.to_csv("mhw_events_2000.csv", index=False)

## MHWs Analysis

In [ ]:
mhw_all = pd.read_csv("mhw_events_2000.csv")

In [ ]:
mhw_all["date_start"] = pd.to_datetime(mhw_all["date_start"], errors="coerce")
mhw_all["date_end"] = pd.to_datetime(mhw_all["date_end"], errors="coerce")

In [ ]:
mhw_all['duration'] = mhw_all['date_end'] - mhw_all['date_start']
mhw_all['duration'].head()

In [ ]:
mhw_all = mhw_all.set_index(["lat", "lon"]).sort_index()
grouped = mhw_all.groupby(level=["lat", "lon"])


In [ ]:
mhw_all.columns

In [ ]:
for (lat, lon), df in grouped:
    print(lat, lon)
    print(df[['date_start', 'date_end','duration', 'category']])
    print()


In [ ]:
mhw_df.info()

## Graphs

In [ ]:
# =================================================
# PREP: MHW catalogue (2000 only)
# =================================================

mhw_df = mhw_all.reset_index()

mhw_df["date_start"] = pd.to_datetime(mhw_df["date_start"])
mhw_df["date_end"] = pd.to_datetime(mhw_df["date_end"])

# =================================================
# SPLIT EVENTS INTO MONTHLY DURATIONS
# =================================================

rows = []

for _, row in mhw_df.iterrows():
    start = row["date_start"]
    end = row["date_end"]
    current = start

    while current <= end:
        month_start = current.replace(day=1)
        next_month = month_start + pd.offsets.MonthBegin(1)
        month_end = min(end, next_month - pd.Timedelta(days=1))

        duration_days = (month_end - current).days + 1

        rows.append({
            "lat": row["lat"],
            "lon": row["lon"],
            "month": current.month,
            "duration_days": duration_days
        })

        current = month_end + pd.Timedelta(days=1)

monthly_events = pd.DataFrame(rows)

# =================================================
# AGGREGATE: sum DURATION PER GRID PER MONTH
# =================================================

monthly_dfs = {}

for month in range(1, 13):
    df_m = monthly_events[monthly_events["month"] == month]

    stats_m = (
        df_m
        .groupby(["lat", "lon"])
        .agg(sum_duration_days=("duration_days", "sum"))
        .reset_index()
    )

    monthly_dfs[month] = stats_m

In [ ]:
monthly_dfs[3]

In [ ]:
# =================================================
# CELL-BASED MONTHLY MAPS (NO INTERPOLATION)
# =================================================

# Base grid
lons = y2000_sel["lon"].values
lats = y2000_sel["lat"].values
lon2d, lat2d = np.meshgrid(lons, lats)

# Ocean mask
ocean_mask = ~np.isnan(y2000_sel["sst"].isel(time=0).values)

# Figure
fig = plt.figure(figsize=(26, 14))
gs = GridSpec(3, 5, width_ratios=[1, 1, 1, 1, 0.05], wspace=0.15, hspace=0.25)

axes = [fig.add_subplot(gs[i // 4, i % 4], projection=ccrs.PlateCarree()) for i in range(12)]
cax = fig.add_subplot(gs[:, -1])

# Color scale
vmin = 0
vmax = max(df["sum_duration_days"].max() for df in monthly_dfs.values() if not df.empty)

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# =================================================
# PLOT EACH MONTH
# =================================================

for ax, month in zip(axes, range(1, 13)):

    # Start with zero everywhere
    grid = np.zeros((len(lats), len(lons)))

    # Fill values for cells with MHWs
    df = monthly_dfs[month]
    for _, row in df.iterrows():
        i = np.where(lats == row["lat"])[0][0]
        j = np.where(lons == row["lon"])[0][0]
        grid[i, j] = row["sum_duration_days"]

    # Mask land only
    grid = np.ma.masked_where(~ocean_mask, grid)

    ax.set_extent(
        [lons.min(), lons.max(), lats.min(), lats.max()],
        crs=ccrs.PlateCarree()
    )

    ax.add_feature(cfeature.COASTLINE, linewidth=1)
    ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.8)

    mesh = ax.pcolormesh(
        lon2d, lat2d, grid,
        cmap="hot_r",
        shading="nearest",   # key line
        vmin=vmin, vmax=vmax,
        transform=ccrs.PlateCarree()
    )

    ax.set_title(month_names[month - 1])

# Colorbar + title
cbar = fig.colorbar(mesh, cax=cax)
cbar.set_label("Total MHW days in month")

fig.suptitle("2000 - Sum of Marine Heatwave Days per Month", fontsize=18, y=0.98)

plt.show()
